# 95 — All-Feature Fusion: 2D + 3D + Bio-FP + Pharmacophore

Combine all feature streams generated in nb87–nb92:
- Combined Morgan+RDKit (2265-dim)
- ChEMBL NR bio-FP (5-dim)
- 3D shape descriptors (12-dim, if nb88 ran)
- PXR pharmacophore features (50-dim)
- Multi-NR transfer prediction (1-dim)

This is the 'kitchen sink' model. LGBM handles high-dimensional sparse input well.
Expected to be a strong individual model and valuable ensemble member.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
# Build idx_active / idx_inactive
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 0


In [4]:
feature_streams = {"combined_2265": X_tr}
labels = {"combined_2265": X_te}

# Load additional feature arrays
for name, fname_tr, fname_te in [
    ("bio_nr_fp",   "bio_fp_tr.npy",      "bio_fp_te.npy"),
    ("3d_shape",    "X_tr_3d.npy",     "X_te_3d.npy"),
    ("tox21_bio",   "tox21_bio_tr.npy",   "tox21_bio_te.npy"),
]:
    p_tr = DATA_PROCESSED/fname_tr; p_te = DATA_PROCESSED/fname_te
    if p_tr.exists() and p_te.exists():
        arr_tr = np.load(p_tr).astype(np.float32)
        arr_te = np.load(p_te).astype(np.float32)
        if arr_tr.shape[0] == len(tr):
            feature_streams[name] = arr_tr
            labels[name] = arr_te
            print(f"Loaded {name}: {arr_tr.shape}")
        else:
            print(f"Skip {name}: wrong shape {arr_tr.shape}")
    else:
        print(f"Not found: {name} ({fname_tr})")

# Multi-NR transfer prediction
for oof_name in ["multi_nr_transfer", "bio_nr_fingerprint"]:
    p = DATA_PROCESSED/f"oof_{oof_name}.npy"
    pt = DATA_PROCESSED/f"te_oof_{oof_name}.npy"
    if p.exists() and pt.exists():
        arr = np.load(p).astype(np.float32).reshape(-1,1)
        arr_te = np.load(pt).astype(np.float32).reshape(-1,1)
        if len(arr) == len(tr):
            feature_streams[oof_name] = arr
            labels[oof_name] = arr_te
            print(f"Loaded {oof_name} prediction: {arr.shape}")

# Fuse all
X_fused_tr = np.hstack(list(feature_streams.values())).astype(np.float32)
X_fused_te = np.hstack(list(labels.values())).astype(np.float32)
# Final impute for any remaining NaNs
X_fused_tr = np.where(np.isfinite(X_fused_tr), X_fused_tr, 0.0)
X_fused_te = np.where(np.isfinite(X_fused_te), X_fused_te, 0.0)
print(f"\nFused feature matrix: {X_fused_tr.shape}")
print(f"Feature streams: {list(feature_streams.keys())}")


Loaded bio_nr_fp: (4139, 5)
Loaded 3d_shape: (4139, 12)
Loaded tox21_bio: (4139, 6)
Loaded multi_nr_transfer prediction: (4139, 1)
Loaded bio_nr_fingerprint prediction: (4139, 1)

Fused feature matrix: (4139, 2290)
Feature streams: ['combined_2265', 'bio_nr_fp', '3d_shape', 'tox21_bio', 'multi_nr_transfer', 'bio_nr_fingerprint']


In [5]:
oof = np.full(len(y_tr), np.nan)
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = lgb.train(LGBM, lgb.Dataset(X_fused_tr[tr_idx], label=y_tr[tr_idx]),
                  valid_sets=[lgb.Dataset(X_fused_tr[va_idx], label=y_tr[va_idx])],
                  callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof[va_idx] = m.predict(X_fused_tr[va_idx])
    print(f"  fold {fold+1}  RAE={rae(y_tr[va_idx], oof[va_idx]):.4f}", flush=True)

m_res = full_metrics(y_tr, oof, cliff_pairs, "all_feature_fusion")
m_res_a = full_metrics(y_tr[active_mask], oof[active_mask], label="fusion [active]")
print("\n" + pd.DataFrame([m_res, m_res_a], index=["overall","active"]).round(4).to_string())

m_final = lgb.train(LGBM, lgb.Dataset(X_fused_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_preds = np.clip(m_final.predict(X_fused_te), y_tr.min()-0.5, y_tr.max()+0.5)
np.save(DATA_PROCESSED/"oof_all_feature_fusion.npy", oof)
np.save(DATA_PROCESSED/"te_oof_all_feature_fusion.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"95_all_feature_fusion.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}  Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")


  fold 1  RAE=0.5015


  fold 2  RAE=0.5932


  fold 3  RAE=0.6046


  fold 4  RAE=0.5776


  fold 5  RAE=0.6121


  [all_feature_fusion] RAE=0.5728 MAE=0.5211 R²=0.5884 r=0.7671 ρ=0.7201 τ=0.5288
  [fusion [active]] RAE=3.8651 MAE=0.8105 R²=-10.4563 r=0.0013 ρ=0.0767 τ=0.0520

            RAE     MAE       R2  Pearson  Spearman  Kendall
overall  0.5728  0.5211   0.5884   0.7671    0.7201   0.5288
active   3.8651  0.8105 -10.4563   0.0013    0.0767   0.0520


Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\95_all_feature_fusion.csv  Test: min=2.40 med=5.02 max=6.16
